# Candidates

In [4]:
import os
import pickle
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp

from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.nearest_neighbours import CosineRecommender

from data_utils import mark_positive, add_weight, get_day_boundaries, build_mappings, build_matrix, build_ground_truth, save_parquet
import polars as pl

from tqdm import tqdm

In [5]:
MODEL_PARAMS = {
    "als": {"factors": 768, "iterations": 15, "regularization": 0.001},
    "ease": {"reg_lambda": 5000, "max_items": 30_000},
    "bpr": {"factors": 1024, "iterations": 600, "learning_rate": 0.05},
    "knn": {"K": 200},
}

BATCH_SIZE = 500
CHECKPOINT_EVERY = 5_000

SPLITS = {
    "train": {"cf_days": [1, 2, 3, 4, 5], "label_day": 6},
    "val": {"cf_days": [1, 2, 3, 4, 5, 6], "label_day": 7},
    "test": {"cf_days": [1, 2, 3, 4, 5, 6, 7], "label_day": None},
}

In [6]:
df_raw = pl.read_parquet("data/train.parquet").to_pandas()
df_raw = mark_positive(df_raw)
df_raw = add_weight(df_raw)
df_raw["day"] = get_day_boundaries(df_raw, "date")

MIN_ITEM_INTERACTIONS = 25
item_counts = df_raw[df_raw["is_positive"]].groupby("item_id")["user_id"].nunique()
popular_items = item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index
df_raw = df_raw[df_raw["item_id"].isin(popular_items)]
print(f"Items after filtering: {len(popular_items)}")

df = df_raw.copy()

mappings = build_mappings(df)
user2idx = mappings["user2idx"]
idx2user = mappings["idx2user"]
item2idx = mappings["item2idx"]
idx2item = mappings["idx2item"]

target_users = pl.read_parquet("data/target_user_ids.parquet").to_pandas()
target_users_all = target_users["user_id"].tolist()

print(f"{len(user2idx)=} {len(item2idx)=}")

Items after filtering: 161211
len(user2idx)=407911 len(item2idx)=161211


## CF Candidates

In [ ]:
def train_als(
    cf_matrix: sp.csr_matrix,
    factors: int = 128,
    iterations: int = 15,
    regularization: float = 0.01
) -> AlternatingLeastSquares:
    model = AlternatingLeastSquares(
        factors=factors, iterations=iterations, regularization=regularization, use_gpu=False
    )
    model.fit(cf_matrix)
    return model


def train_bpr(
    cf_matrix: sp.csr_matrix,
    factors: int = 128,
    iterations: int = 15,
    learning_rate: float = 0.05
) -> BayesianPersonalizedRanking:
    model = BayesianPersonalizedRanking(
        factors=factors,
        iterations=iterations,
        learning_rate=learning_rate,
        use_gpu=False
    )
    model.fit(cf_matrix)
    return model


def train_knn(cf_matrix: sp.csr_matrix, K: int = 200) -> CosineRecommender:
    model = CosineRecommender(K=K)
    model.fit(cf_matrix)
    return model


def train_ease(
    cf_matrix: sp.csr_matrix,
    reg_lambda: float = 500,
    max_items: int = 30_000
) -> tuple[np.ndarray, np.ndarray, sp.csr_matrix]:
    item_counts = np.diff(cf_matrix.T.tocsr().indptr)
    if cf_matrix.shape[1] > max_items:
        top_item_idxs = np.argsort(-item_counts)[:max_items]
    else:
        top_item_idxs = np.arange(cf_matrix.shape[1])
    ease_sub = cf_matrix[:, top_item_idxs]
    X = ease_sub.toarray().astype(np.float32)
    G = X.T @ X
    G += reg_lambda * np.eye(G.shape[0], dtype=np.float32)
    P = np.linalg.inv(G)
    B = P / (-np.diag(P))
    np.fill_diagonal(B, 0.0)
    return B, top_item_idxs, ease_sub


def train_models(
    cf_matrix: sp.csr_matrix,
) -> tuple[AlternatingLeastSquares, BayesianPersonalizedRanking, CosineRecommender, np.ndarray, np.ndarray, sp.csr_matrix]:
    print("Training ALS")
    als = train_als(cf_matrix, **MODEL_PARAMS["als"])

    print("Training BPR")
    bpr = train_bpr(cf_matrix, **MODEL_PARAMS["bpr"])

    print("Training KNN")
    knn = train_knn(cf_matrix, **MODEL_PARAMS["knn"])

    print("Training EASE")
    ease_B, ease_top_idxs, ease_sub = train_ease(cf_matrix, **MODEL_PARAMS["ease"])

    return als, bpr, knn, ease_B, ease_top_idxs, ease_sub

In [ ]:
def make_rank_score_dicts(iidxs_row: np.ndarray, scores_row: np.ndarray, K: int) -> tuple[dict[int, int], dict[int, float]]:
    rank_d, score_d = {}, {}
    for j, idx in enumerate(iidxs_row):
        idx = int(idx)
        if idx >= 0:
            rank_d[idx] = j
            score_d[idx] = float(scores_row[j])
    return rank_d, score_d

    
def get_target_users(label_day: int | None, df: pd.DataFrame, user2idx: dict[int, int]) -> list[int]:
    if label_day is not None:
        ground_truth = build_ground_truth(df, label_day, user2idx)
        all_target = [u for u in ground_truth if u in user2idx]
    else:
        all_target = [u for u in target_users_all if u in user2idx]
    print(f"Target users: {len(all_target)}")
    return all_target


def get_pop_scores(cf_matrix: sp.csr_matrix) -> np.ndarray:
    item_pop = np.diff(cf_matrix.T.tocsr().indptr).astype(float)
    n_items_all = cf_matrix.shape[1]
    pop_order = np.argsort(-item_pop)
    pop_rank_arr = np.empty(n_items_all, dtype=np.int32)
    pop_rank_arr[pop_order] = np.arange(n_items_all)
    pop_score_arr = 1.0 - pop_rank_arr / n_items_all
    return pop_score_arr

In [ ]:
def generate_cf_candidates(split_name: str) -> pd.DataFrame:
    cfg = SPLITS[split_name]
    cf_days = cfg["cf_days"]
    label_day = cfg["label_day"]
    final_path = f"data/processed/cf_candidates_{split_name}.parquet"
    partial_path = f"data/processed/cf_candidates_{split_name}_partial.parquet"

    if os.path.exists(final_path):
        print(f"Candidates already done {final_path}")
        return pd.read_parquet(final_path)
    
    cf_data = df[df["day"].isin(cf_days)].copy()
    cf_matrix = build_matrix(cf_data, mappings, binary=False)
    print(f"cf_matrix {cf_matrix.shape}")

    als, bpr, knn, ease_B, ease_top_idxs, ease_sub = train_models(cf_matrix)

    all_target = get_target_users(label_day, df, user2idx)
    pop_score_arr = get_pop_scores(cf_matrix)

    seen_per_user = (
        cf_data
        .groupby("user_id")["item_id"]
        .apply(lambda items: {item2idx[i] for i in items if i in item2idx})
        .to_dict()
    )

    done_users = set()
    rows = []

    if os.path.exists(partial_path):
        partial_df = pd.read_parquet(partial_path)
        done_users = set(partial_df["user_id"].unique())
        rows = partial_df.to_dict("records")
    
    remaining = [u for u in all_target if u not in done_users]
    n_since_ckpt = 0

    for start in tqdm(range(0, len(remaining), BATCH_SIZE)):
        batch_users = remaining[start:start + BATCH_SIZE]
        uidxs = np.array([user2idx[u] for u in batch_users])

        als_ii, als_ss = als.recommend(uidxs, cf_matrix[uidxs], N=80, filter_already_liked_items=True)
        bpr_ii, bpr_ss = bpr.recommend(uidxs, cf_matrix[uidxs], N=80, filter_already_liked_items=True)
        knn_ii, knn_ss = knn.recommend(uidxs, cf_matrix[uidxs], N=50, filter_already_liked_items=True)

        ease_sc = np.asarray(ease_sub[uidxs].dot(ease_B))
        ease_sc[ease_sub[uidxs].toarray().astype(bool)] = -np.inf
        ease_topk = np.argpartition(-ease_sc, 80, axis=1)[:, :80]

        for i, user_id in enumerate(batch_users):
            als_rank, als_score = make_rank_score_dicts(als_ii[i], als_ss[i], 80)
            bpr_rank, bpr_score = make_rank_score_dicts(bpr_ii[i], bpr_ss[i], 80)
            knn_rank, knn_score = make_rank_score_dicts(knn_ii[i], knn_ss[i], 50)

            ease_pos = ease_topk[i]
            ease_row_sc = ease_sc[i, ease_pos]
            ease_order = np.argsort(-ease_row_sc)
            ease_orig = ease_top_idxs[ease_pos[ease_order]]
            ease_row_sc = ease_row_sc[ease_order]
            ease_rank = {int(ease_orig[j]): j for j in range(len(ease_orig))}
            ease_score = {int(ease_orig[j]): float(ease_row_sc[j]) for j in range(len(ease_orig))}

            user_seen = seen_per_user.get(user_id, set())
            all_cands = (set(als_rank) | set(bpr_rank) | set(knn_rank) | set(ease_rank)) - user_seen

            for item_idx in all_cands:
                a_r = als_rank.get(item_idx, 80)
                a_s = als_score.get(item_idx, 0.0)
                
                b_r = bpr_rank.get(item_idx, 80)
                b_s = bpr_score.get(item_idx, 0.0)
                
                k_r = knn_rank.get(item_idx, 50)
                k_s = knn_score.get(item_idx, 0.0)
                
                e_r = ease_rank.get(item_idx, 80)
                e_s = ease_score.get(item_idx, 0.0)
                
                ranks = [a_r, b_r, k_r, e_r]
                rows.append({
                    "user_id": user_id,
                    "item_id": idx2item[item_idx],
                    "als_score": a_s,
                    "bpr_score": b_s,
                    "knn_score": k_s,
                    "ease_score": e_s,
                    "pop_score": float(pop_score_arr[item_idx]),
                    "n_models_recommended": int(a_r < 80) + int(b_r < 80) + int(k_r < 50) + int(e_r < 80),
                    "best_rank": min(a_r, b_r, k_r, e_r),
                    "mean_rank": (a_r + b_r + k_r + e_r) / 4,
                    "rank_std": float(np.std(ranks)),
                    "n_models_top50": sum(r < 50 for r in ranks),
                })

        n_since_ckpt += len(batch_users)
        if n_since_ckpt >= CHECKPOINT_EVERY:
            save_parquet(pd.DataFrame(rows), partial_path)
            total_done = len(done_users) + start + len(batch_users)
            print(f"Checkpoint: {total_done}/{len(all_target)}")
            n_since_ckpt = 0

    result_df = pd.DataFrame(rows)
    save_parquet(result_df, final_path)

    if os.path.exists(partial_path):
        os.remove(partial_path)

    return result_df

In [10]:
cands_train = generate_cf_candidates("train")

Candidates already done data/processed/cf_candidates_train.parquet


In [11]:
cands_val = generate_cf_candidates("val")

Candidates already done data/processed/cf_candidates_val.parquet


In [12]:
cands_test = generate_cf_candidates("test")

Candidates already done data/processed/cf_candidates_test.parquet


## Features

In [ ]:
def last_n(cf_days: list[int], n: int) -> list[int]:
    return sorted(cf_days)[-n:]


def compute_user_features(cf_data: pd.DataFrame, cf_days: list[int]) -> pd.DataFrame:
    d1 = cf_data[cf_data["day"].isin(last_n(cf_days, 1))]
    d2 = cf_data[cf_data["day"].isin(last_n(cf_days, 2))]
    d3 = cf_data[cf_data["day"].isin(last_n(cf_days, 3))]

    pos1 = d1[d1["is_positive"]]
    pos2 = d2[d2["is_positive"]]
    pos3 = d3[d3["is_positive"]]

    watches_all = cf_data[cf_data["event_type"] == "watch_time"]
    watches3 = d3[d3["event_type"] == "watch_time"]
    long3 = watches3[watches3["watch_time"] > 60]
    short3 = watches3[watches3["watch_time"] <= 60]

    feats = pd.DataFrame({
        "u_positive_events_last_1_days": pos1.groupby("user_id").size(),
        "u_positive_events_last_2_days": pos2.groupby("user_id").size(),
        "u_positive_events_last_3_days": pos3.groupby("user_id").size(),
        "u_all_events_last_1_days": d1.groupby("user_id").size(),
        "u_all_events_last_2_days": d2.groupby("user_id").size(),
        "u_all_events_last_3_days": d3.groupby("user_id").size(),
        "u_avg_timespent": watches_all.groupby("user_id")["watch_time"].mean(),
        "u_n_unique_items_last_3_days": d3.groupby("user_id")["item_id"].nunique(),
    }).fillna(0)

    likes3 = d3[d3["event_type"] == "like"].groupby("user_id").size()
    favs3 = d3[d3["event_type"] == "favorite"].groupby("user_id").size()
    long3_u = long3.groupby("user_id").size()
    short3_u = short3.groupby("user_id").size()

    idx = feats.index
    pos3_cnt = feats["u_positive_events_last_3_days"].clip(lower=1)
    all3_cnt = feats["u_all_events_last_3_days"].clip(lower=1)
    watch3_cnt = (long3_u.reindex(idx).fillna(0) + short3_u.reindex(idx).fillna(0)).clip(lower=1)

    feats["u_like_rate_last_3_days"] = likes3.reindex(idx).fillna(0) / pos3_cnt
    feats["u_fav_rate_last_3_days"] = favs3.reindex(idx).fillna(0) / pos3_cnt
    feats["u_skip_rate_last_3_days"] = short3_u.reindex(idx).fillna(0) / watch3_cnt
    feats["u_positive_rate_last_3_days"] = pos3_cnt / all3_cnt
    feats["u_activity_trend"] = (
        feats["u_all_events_last_1_days"] /
        (feats["u_all_events_last_3_days"] / 3).clip(lower=1)
    )

    return feats.reset_index().rename(columns={"index": "user_id"})


def compute_item_features(cf_data: pd.DataFrame, cf_days: list[int]) -> pd.DataFrame:
    d1 = cf_data[cf_data["day"].isin(last_n(cf_days, 1))]
    d2 = cf_data[cf_data["day"].isin(last_n(cf_days, 2))]
    d3 = cf_data[cf_data["day"].isin(last_n(cf_days, 3))]
    df_first2 = cf_data[cf_data["day"].isin(sorted(cf_days)[:2])]

    pos1 = d1[d1["is_positive"]]
    pos2 = d2[d2["is_positive"]]
    pos3 = d3[d3["is_positive"]]
    pos_first2 = df_first2[df_first2["is_positive"]]

    watches_all = cf_data[cf_data["event_type"] == "watch_time"]
    watches3 = d3[d3["event_type"] == "watch_time"]
    long3 = watches3[watches3["watch_time"] > 60]
    short3 = watches3[watches3["watch_time"] <= 60]

    feats = pd.DataFrame({
        "i_positive_events_last_1_days": pos1.groupby("item_id").size(),
        "i_positive_events_last_2_days": pos2.groupby("item_id").size(),
        "i_positive_events_last_3_days": pos3.groupby("item_id").size(),
        "i_unique_users_last_1_days": pos1.groupby("item_id")["user_id"].nunique(),
        "i_unique_users_last_2_days": pos2.groupby("item_id")["user_id"].nunique(),
        "i_unique_users_last_3_days": pos3.groupby("item_id")["user_id"].nunique(),
        "i_avg_timespent": watches_all.groupby("item_id")["watch_time"].mean(),
    }).fillna(0)

    likes3 = d3[d3["event_type"] == "like"].groupby("item_id").size()
    favs3 = d3[d3["event_type"] == "favorite"].groupby("item_id").size()
    long3_i = long3.groupby("item_id").size()
    short3_i = short3.groupby("item_id").size()

    idx = feats.index
    pos3_cnt = feats["i_positive_events_last_3_days"].clip(lower=1)
    uniq3_cnt = feats["i_unique_users_last_3_days"].clip(lower=1)
    watch3_cnt = (long3_i.reindex(idx).fillna(0) + short3_i.reindex(idx).fillna(0)).clip(lower=1)

    feats["i_like_rate_last_3_days"] = likes3.reindex(idx).fillna(0) / pos3_cnt
    feats["i_fav_rate_last_3_days"] = favs3.reindex(idx).fillna(0) / pos3_cnt
    feats["i_completion_rate_last_3_days"] = long3_i.reindex(idx).fillna(0) / watch3_cnt
    feats["i_skip_rate_last_3_days"] = short3_i.reindex(idx).fillna(0) / watch3_cnt
    feats["i_interactions_per_user_last_3"] = pos3_cnt / uniq3_cnt
    feats["i_user_growth"] = (
        feats["i_unique_users_last_1_days"] /
        (feats["i_unique_users_last_3_days"] / 3).clip(lower=1)
    )

    f2 = pos_first2.groupby("item_id").size()
    l2 = pos2.groupby("item_id").size()
    feats["i_trend"] = (l2.reindex(idx).fillna(0) + 1) / (f2.reindex(idx).fillna(0) + 1)

    vals = feats["i_unique_users_last_3_days"].values
    rank_arr = np.empty(len(feats), dtype=np.int32)
    rank_arr[np.argsort(-vals)] = np.arange(len(feats))
    feats["i_popularity_rank_last_3_days"] = rank_arr

    return feats.reset_index().rename(columns={"index": "item_id"})


def set_label(label_day: int | None, df: pd.DataFrame, user2idx: dict[int, int], cands: pd.DataFrame) -> pd.DataFrame:
    if label_day is not None:
        ground_truth = build_ground_truth(df, label_day, user2idx)
        gt_df = pd.DataFrame(
            [(u, i) for u, items in ground_truth.items() for i in items],
            columns=["user_id", "item_id"],
        ).assign(label=np.int8(1))
        result = cands.merge(gt_df, on=["user_id", "item_id"], how="left")
        result["label"] = result["label"].fillna(0).astype(np.int8)
        return result
    
    result = cands.copy()
    result["label"] = pd.array([pd.NA] * len(result))
    return result

In [ ]:
def build_and_save_l2r(split_name: str, cands: pd.DataFrame) -> pd.DataFrame:
    out_path = f"data/processed/{split_name}_l2r.parquet"

    cfg = SPLITS[split_name]
    cf_days = cfg["cf_days"]
    label_day = cfg["label_day"]
    cf_data = df[df["day"].isin(cf_days)].copy()
    
    basis = set_label(label_day, df, user2idx, cands)
    u_feats = compute_user_features(cf_data, cf_days)
    i_feats = compute_item_features(cf_data, cf_days)

    result = basis.merge(u_feats, on="user_id", how="left").merge(i_feats, on="item_id", how="left")

    feat_cols = [c for c in result.columns if c.startswith(("u_", "i_"))]
    result[feat_cols] = result[feat_cols].fillna(0)

    for col in ["als_score", "bpr_score", "ease_score"]:
        user_max = result.groupby("user_id")[col].transform("max").clip(lower=1e-9)
        result[f"{col}_norm"] = (result[col] / user_max).astype(np.float32)

    save_parquet(result, out_path)

    print(f"L2R saved: {len(result)} rows")

    return result

In [ ]:
train_l2r = build_and_save_l2r("train", cands_train)

stat = train_l2r.groupby("user_id")["label"].sum()
allowed_users = stat[stat >= 1].index

save_parquet(train_l2r[train_l2r.user_id.isin(allowed_users)], "data/processed/train_l2r.parquet")

In [ ]:
val_l2r = build_and_save_l2r("val", cands_val)

In [ ]:
test_l2r = build_and_save_l2r("test", cands_test)

## Summary

In [29]:
for name, dataset in [("train", train_l2r), ("val", val_l2r), ("test", test_l2r)]:
    print("="*40)
    print(f"{name=}")
    print(f"shape: {dataset.shape}")
    print(f"avg candidates / user: {dataset.groupby('user_id').size().mean()}")

    if dataset["label"].notna().any():
        n_pos   = (dataset["label"] == 1).sum()
        n_total = len(dataset)
        print(f"positives: {n_pos} / {n_total} ({n_pos/n_total})")

    feat_cols = [c for c in dataset.columns if c.startswith(("u_", "i_"))]
    nan_cnt = dataset[feat_cols].isna().sum().sum()
    print(f"NaN count: {nan_cnt}")

name='train'
shape: (61232272, 44)
avg candidates / user: 241.686620302028
positives: 181772 / 61232272 (0.0029685653343060666)
NaN count: 0
name='val'
shape: (48939251, 44)
avg candidates / user: 239.77135144139376
positives: 129364 / 48939251 (0.002643358804163145)
NaN count: 0
name='test'
shape: (47969544, 44)
avg candidates / user: 239.70509546819642
NaN count: 0


In [24]:
for name, dataset, label_d in [("train", train_l2r, 6), ("val", val_l2r, 7)]:
    gt = build_ground_truth(df, label_d, user2idx)
    total_gt = sum(len(v) for v in gt.values())
    found = int((dataset["label"] == 1).sum())
    print(f"{name}: {found} / {total_gt} = {found / max(total_gt, 1)}")

train: 181772 / 1960099 = 0.09273613220556717
val: 129364 / 1311117 = 0.0986670144617147
